In [1]:
# セル1: モジュール＋データロード
import numpy as np
import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embedding_matrix = np.load('70_embeddings.npy')
with open('71_sst_data.pkl','rb') as f:
    d = pickle.load(f)
train_data, dev_data = d['train'], d['dev']


In [2]:
# セル2: collate＋MLPClassifier 定義
def collate(batch):
    batch = sorted(batch, key=lambda x: x['input_ids'].size(0), reverse=True)
    xs = [ex['input_ids'] for ex in batch]
    ys = [ex['label']     for ex in batch]
    x_pad = pad_sequence(xs, batch_first=True, padding_value=0)
    y_cat = torch.cat(ys).view(-1,1)
    return x_pad, y_cat

class MLPClassifier(nn.Module):
    def __init__(self, emb_matrix, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix, dtype=torch.float),
            freeze=False, padding_idx=0
        )
        emb_dim = emb_matrix.shape[1]
        self.fc1 = nn.Linear(emb_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb  = self.embedding(input_ids)
        mask = (input_ids!=0).unsqueeze(-1).float()
        s    = (emb*mask).sum(dim=1)
        l    = mask.sum(dim=1).clamp(min=1)
        avg  = s / l
        h    = self.act(self.fc1(avg))
        logit= self.fc2(h).squeeze(-1)
        return torch.sigmoid(logit)


In [3]:
# セル2: collate＋MLPClassifier 定義
def collate(batch):
    batch = sorted(batch, key=lambda x: x['input_ids'].size(0), reverse=True)
    xs = [ex['input_ids'] for ex in batch]
    ys = [ex['label']     for ex in batch]
    x_pad = pad_sequence(xs, batch_first=True, padding_value=0)
    y_cat = torch.cat(ys).view(-1,1)
    return x_pad, y_cat

class MLPClassifier(nn.Module):
    def __init__(self, emb_matrix, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix, dtype=torch.float),
            freeze=False, padding_idx=0
        )
        emb_dim = emb_matrix.shape[1]
        self.fc1 = nn.Linear(emb_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb  = self.embedding(input_ids)
        mask = (input_ids!=0).unsqueeze(-1).float()
        s    = (emb*mask).sum(dim=1)
        l    = mask.sum(dim=1).clamp(min=1)
        avg  = s / l
        h    = self.act(self.fc1(avg))
        logit= self.fc2(h).squeeze(-1)
        return torch.sigmoid(logit)


In [4]:
# セル3: 訓練＋評価
model     = MLPClassifier(embedding_matrix).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
loader    = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate)

# 学習
model.train()
for epoch in range(3):
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y.squeeze())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch} done")

# 評価
model.eval()
correct = 0
with torch.no_grad():
    for i in range(0, len(dev_data), 32):
        batch = dev_data[i:i+32]
        x, y = collate(batch)
        x, y = x.to(device), y.to(device)
        correct += (model(x).round().squeeze() == y.squeeze()).sum().item()
acc = correct / len(dev_data)
print(f"Dev Accuracy (MLP): {acc:.4f}")


Epoch 0 done
Epoch 1 done
Epoch 2 done
Dev Accuracy (MLP): 0.7913
